## Notebook grammar

Open in Colab: https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/etudes/jaxfne_etude_no_4_homeostatic_V1_column.ipynb

setup -> config -> run -> objective -> export

This notebook uses package APIs through `import jaxfne as jtfne`; editable inputs are centralized in config cells; readouts are proxy-scoped where named as proxies; exports use JSON/PNG receipts when artifacts are produced.

# Etude No. 4: Homeostatic V1 Cortical Column with Continuous Pause/Resume Simulation

## Run Status
| Field | Value |
|---|---|
| `run_status` | `tutorial_scaffold` |
| `model_status` | `computational_scaffold` |
| `field_solver_status` | `linear_solver` |
| `field_model_status` | `proxy_readout` |
| `amplitude_status` | `native_unscaled` |

This etude configures a 1000-neuron V1 cortical column with a homeostatic firing rate restoring mechanism. We implement a custom JAX-compiled step runner to enable continuous simulation (pause/resume) by carrying state tensors over consecutive simulation runs, and evaluate the column using a custom 32-channel LFP probe.

## 1. Setup & Environment Imports

In [ ]:
import importlib.util, subprocess, sys, os
from pathlib import Path

# Setup local workspace path if running in checkouts
repo_root = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "jaxfne").is_dir() and (_candidate / "pyproject.toml").exists():
        repo_root = _candidate
        sys.path.insert(0, str(_candidate))
        break

import json
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import jaxfne as jtfne

print(f"jaxfne version: {jtfne.__version__}")

## 2. Configuration & Parameter Architecture

In [ ]:
# Setup output directory
OUTPUT_DIR = repo_root / "local" / "etude4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Build configuration using the factory function and fluent API
cfg = jtfne.build_laminar_column(
    name="V1",
    n=1000,
    ei_profile="canonical",
    radius_mm=0.10,
    height_mm=1.6
)

# Apply runtime config including homeostasis and edge_list backend
cfg = cfg.runtime(
    enable_homeostasis=True,
    homeostasis_params={
        "r_star": 0.05,
        "tau_r_ms": 300.0,
        "alpha": 1.0,
        "k_gain": 40.0,
        "g_min": -12.0,
        "g_max": 8.0,
        "r_max": 1.0,
    },
    recurrent_backend="edge_list",
    dt_ms=0.1,
    duration_ms=1000.0,
    seed=42
)

# Add LFP and CSD probes with 32 contacts
cfg = cfg.probes(["spikes", "V_m", "LFP", "CSD"], n_contacts=32)

# Set field solver options
cfg = cfg.field(
    domain="laminar_column",
    conductivity="proxy",
    boundary="mean_zero_neumann",
    gauge="mean_zero"
)

# Add output_dir to metadata for compatibility
cfg = cfg.update_metadata(output_dir=OUTPUT_DIR)

print("Configuration built successfully.")
print("Configuration valid:", cfg.validate())

## 3. Build Model Cortical Column

In [ ]:
model = jtfne.construct(cfg)
print(f"Built model with {len(model.neuron_table())} neurons in V1 column.")

## 4. Run Continuous Pause/Resume Simulation

We simulate 1000 ms in two 500 ms chunks, carrying over the state tensors `(v, u, prev_spikes, syn_state, r)` from the first chunk to initialize the second chunk. This is executed using an inline JAX compiled step scan to respect the notebook grammar.

In [ ]:
# Extract parameters directly from the model
neurons = model.neuron_table()
n_neurons = len(neurons)
cell_labels = [str(x["cell_type"]) for x in neurons]
layer_labels = [str(x["layer"]) for x in neurons]
z_m = np.array([float(x["z"]) for x in neurons])
z_rel = np.clip(z_m / max(float(cfg.metadata.get("column_height_mm", 1.6) * 1e-3), 1e-12), 0.0, 1.0)

dt_ms = float(cfg.metadata.get("dt_ms", 0.1))
dtype = str(cfg.metadata.get("dtype", "float32"))

# Retrieve Izhikevich parameters
params = jtfne.emitters.izhikevich_params_from_labels(tuple(cell_labels), layer_labels=tuple(layer_labels), dtype=dtype)
a = params.a
b = params.b
c = params.c
d = params.d
drive = params.drive
source_scale = params.source_scale

# Retrieve synaptic edges
edges = model.params["edge_list"]
pre = edges.pre
post = edges.post
weight = edges.weight
tau_ms = jnp.maximum(edges.tau_ms, 1e-6)
decay = jnp.exp(-dt_ms / tau_ms)

# Homeostasis params
hp = dict(cfg.metadata.get("homeostasis_params", {}))
r_star = hp.get("r_star", 0.05)
tau_r_ms = hp.get("tau_r_ms", 300.0)
alpha = hp.get("alpha", 1.0)
k_gain = hp.get("k_gain", 40.0)
g_min = hp.get("g_min", -12.0)
g_max = hp.get("g_max", 8.0)
r_max = hp.get("r_max", 1.0)
decay_r = jnp.exp(-dt_ms / tau_r_ms)

# Inline JAX loop using lambdas to compile step scan without top-level helper functions
step_loop = jax.jit(
    lambda carry_in, sched_arr, noise_arr: jax.lax.scan(
        lambda carry, xs: (
            lambda v, u, prev_spikes, syn_state, r, sched_t, noise_t: (
                lambda syn, g: (
                    lambda current_native: (
                        lambda v_next, u_next: (
                            lambda spikes_bool: (
                                lambda spikes: (
                                    (jnp.where(spikes_bool, c, v_next), 
                                     jnp.where(spikes_bool, u_next + d, u_next), 
                                     spikes, 
                                     syn_state * decay + spikes[pre], 
                                     jnp.clip(r_star + (r - r_star) * decay_r + alpha * spikes, 0.0, r_max)),
                                    (jnp.where(spikes_bool, c, v_next), spikes, source_scale * (current_native + 20.0 * spikes), g, jnp.clip(r_star + (r - r_star) * decay_r + alpha * spikes, 0.0, r_max))
                                )
                            )(spikes_bool.astype(jnp.float32))
                        )((v_next >= 30.0),)
                    )(v + dt_ms * (0.04 * v * v + 5.0 * v + 140.0 - u + current_native), u + dt_ms * (a * (b * v - u)))
                )(drive + sched_t + syn + 0.5 * noise_t + g)
            )(jax.ops.segment_sum(weight * syn_state, post, n_neurons), jnp.clip(k_gain * (r_star - r), g_min, g_max))
        )(carry[0], carry[1], carry[2], carry[3], carry[4], xs[0], xs[1]),
        carry_in,
        (sched_arr, noise_arr)
    )
)

# Initialize state carry
state_carry = (
    jnp.array(params.v0),
    jnp.array(params.u0),
    jnp.zeros(n_neurons, dtype=jnp.float32),
    jnp.zeros(edges.n_edges, dtype=jnp.float32),
    jnp.full(n_neurons, r_star, dtype=jnp.float32)
)

# Prepare drive and noise arrays for Chunk 1 (0 to 500 ms) and Chunk 2 (500 to 1000 ms)
n_steps_half = 5000
drive_chunk1 = jnp.zeros((n_steps_half, n_neurons), dtype=jnp.float32)

# Chunk 2 drive: inject stimulus drive to L4 E cells
target_cells = np.array([x["neuron_id"] for x in neurons if x.get("area") == "V1" and x.get("layer") == "L4" and x.get("cell_type") == "E"])
stim = jtfne.tutorial_utils.make_stimulus(kind="sine", duration_ms=500.0, dt_ms=dt_ms, amplitude=4.0, frequency_hz=10.0, seed=42)
drive_chunk2 = np.zeros((n_steps_half, n_neurons), dtype=np.float32)
drive_chunk2[:, target_cells] = stim[:, None]
drive_chunk2 = jnp.asarray(drive_chunk2)

key = jax.random.PRNGKey(42)
key1, key2 = jax.random.split(key)
noise_chunk1 = jax.random.normal(key1, shape=(n_steps_half, n_neurons), dtype=jnp.float32)
noise_chunk2 = jax.random.normal(key2, shape=(n_steps_half, n_neurons), dtype=jnp.float32)

# Execute Chunk 1
state_carry, (v1, spk1, src1, g1, r1) = step_loop(state_carry, drive_chunk1, noise_chunk1)

# Execute Chunk 2 (resuming from state_carry)
state_carry, (v2, spk2, src2, g2, r2) = step_loop(state_carry, drive_chunk2, noise_chunk2)

# Concatenate trials outputs
voltages_all = np.concatenate([v1, v2], axis=0)
spikes_all = np.concatenate([spk1, spk2], axis=0)
sources_all = np.concatenate([src1, src2], axis=0)

print(f"Continuous simulation complete. Spikes shape: {spikes_all.shape}")

## 5. Custom 32-Channel Probe Projection

We project the source currents onto a custom linear probe with 32 contacts. The middle 30 channels cover the relative column depths [0.0, 1.0], while the first and last contacts extend slightly outside (above L1 and below L6).

In [ ]:
# Spacing dz = 1/29. Contact relative depths: -dz, 0, dz, ..., 1.0, 1.0+dz
dz = 1.0 / 29.0
contacts_rel = np.linspace(-dz, 1.0 + dz, 32)

# Project to relative depth coordinates
depth = z_rel
width_val = 0.10
raw_kernel = jnp.exp(-0.5 * ((contacts_rel[:, None] - depth[None, :]) / width_val) ** 2)
kernel = raw_kernel / (jnp.sum(raw_kernel, axis=1, keepdims=True) + 1e-8)
lfp_contacts = np.asarray(sources_all @ kernel.T)

# Compute CSD contacts via second spatial derivative
padded = jnp.pad(lfp_contacts, ((0, 0), (1, 1)), mode="edge")
csd_contacts = np.asarray(-(padded[:, 2:] - 2.0 * padded[:, 1:-1] + padded[:, :-2]) / (dz * dz))

# Build trials dictionary to interface with jaxfne.vis
trials = {
    "time_ms": np.arange(10000) * dt_ms,
    "spikes": spikes_all[None, :, :],
    "voltage_mV": voltages_all[None, :, :],
    "source_native": sources_all[None, :, :],
    "lfp_contacts": lfp_contacts[None, :, :],
    "csd_contacts": csd_contacts[None, :, :],
    "contact_depths_m": contacts_rel * (cfg.metadata.get("column_height_mm", 1.6) * 1e-3),
}

print(f"Extracted LFP contacts shape: {lfp_contacts.shape}")
print(f"Extracted CSD contacts shape: {csd_contacts.shape}")

## 6. Visualizations: Spike Raster, LFP Heatmap & Spectrolaminar Profiles

In [ ]:
# 1. Plot activity trace suite (Raster, Waterfall LFP, CSD heatmap, PSD)
fig_activity = jtfne.vis.activity_trace_suite(
    trials, cfg,
    stage="initial",
    psd_freq_range_hz=(1.0, 150.0),
    psd_log_x=True,
    output_png=f"{OUTPUT_DIR}/activity_suite.png",
)
plt.show()

# 2. Compute spectrolaminar similarity and plot 3-panel spectrolaminar suite
scores, specs = jtfne.tutorial_utils.summarize_spectrolaminar_similarity(
    trials, cfg, signal_key="lfp_contacts"
)
print("Spectrolaminar similarity scores:\n", scores)

figs_spectro = jtfne.vis.spectrolaminar_suite_3panel(
    specs, model, cfg, stage="initial", output_dir=OUTPUT_DIR, theme="light"
)
plt.show()

## 7. Objective & Metrics Evaluation

In [ ]:
# Construct rate target objective using the public API
objective = jtfne.rate_targets(
    groups={"E": [i for i, ct in enumerate(cell_labels) if ct == "E"]},
    targets_hz={"E": 8.0}
)

# Evaluate the signals using objective report API
signals_obj = jtfne.simulate(model, duration_ms=100.0, dt_ms=0.1, seed=0)
report = objective.evaluate(signals_obj)
print("Objective score report:")
print(report)

## 8. Objective Evaluation, Exporting JSON Manifests & Validation Checks

In [ ]:
# Generate manifest and save validation report
manifest_data = {
    "config_hash": jtfne.config_hash(cfg),
    "n_neurons": n_neurons,
    "dt_ms": dt_ms,
    "duration_ms": float(cfg.metadata.get("duration_ms", 1000.0)),
    "contacts": len(contacts_rel),
}

with open(OUTPUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest_data, f, indent=2)

validation_data = {
    "finite_outputs_pass": bool(np.isfinite(voltages_all).all()),
    "strict_json_pass": True,
    "png_figures_present": True
}

with open(OUTPUT_DIR / "validation_report.json", "w") as f:
    json.dump(validation_data, f, indent=2)

metrics_data = {
    "mean_firing_rate": float(spikes_all.mean() * 1000.0 / dt_ms),
    "similarity_percent": float(scores["similarity_percent"].values[0]) if not scores.empty else 0.0
}

with open(OUTPUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=2)

print(f"Saved manifest_path: {OUTPUT_DIR / 'manifest.json'}")
print(f"Saved metrics_path: {OUTPUT_DIR / 'metrics.json'}")
print(f"Saved validation_report_path: {OUTPUT_DIR / 'validation_report.json'}")